In [36]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd
import json

# -------------------------- Data Preparation --------------------------
df = pd.DataFrame({
    "id": [101, 102, 103, 104, 105, 106, 107, 108],
    "Name": ["Ali", "Sara", "Reza", "Maryam", "Tom", "Zhina", "Boshra", "Jack"],
    "Age": [20, 31, 28, 24, 26, 33, 21, 29],
    "Score": [80, 95, 76, 88, 92, 70, 85, 99]
})

# Create a rich hover text that includes both Name and ID
df['hover_text'] = df.apply(
    lambda row: f"{row['Name']} (ID: {row['id']})", axis=1
)

# -------------------------- Figure Creation --------------------------
fig = px.scatter(
    df,
    x="Age",
    y="Score",
    custom_data=["id", "Name"],      # Important: custom_data will be available in click/hover/selected events
    hover_name="hover_text",         # This shows in the hover tooltip
    title="Age vs Score Scatter Plot",
    labels={"Age": "Age", "Score": "Score"}
)

# Customize hover template for better readability
fig.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>Age: %{x}<br>Score: %{y}<extra></extra>"
)

# -------------------------- Dash App Setup --------------------------
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Plotly Dash App - Hover, Click & Selected Events with Custom Data"),
    
    dcc.Graph(id='scatter-plot', figure=fig),
    
    # Three columns for event data
    html.Div([
        html.H3("🔍 Hover Data (Raw JSON)"),
        html.Pre(id='hover-data', style={'background': '#f4f4f4', 'padding': '10px', 'border-radius': '5px'}),
    ], style={'width': '32%', 'display': 'inline-block', 'vertical-align': 'top'}),
    
    html.Div([
        html.H3("🖱️ Click Data (Raw JSON)"),
        html.Pre(id='click-data', style={'background': '#f4f4f4', 'padding': '10px', 'border-radius': '5px'}),
    ], style={'width': '32%', 'display': 'inline-block', 'vertical-align': 'top'}),
    
    html.Div([
        html.H3("📦 Selected Data (Raw JSON)"),
        html.Pre(id='selected-data', style={'background': '#f4f4f4', 'padding': '10px', 'border-radius': '5px'}),
    ], style={'width': '32%', 'display': 'inline-block', 'vertical-align': 'top'}),
    
    html.Hr(),
    
    # Human readable section
    html.H3("📋 Human Readable Information"),
    html.Div(id='full-info', 
             style={'padding': '15px', 'background': '#e8f5e9', 'border-radius': '8px', 'margin-top': '10px'})
])

# -------------------------- Callbacks --------------------------

@app.callback(Output('hover-data', 'children'), Input('scatter-plot', 'hoverData'))
def display_hover(hoverData):
    """Display raw hover event data including custom_data."""
    if not hoverData:
        return "No hover event yet."
    return json.dumps(hoverData, indent=2, ensure_ascii=False)


@app.callback(Output('click-data', 'children'), Input('scatter-plot', 'clickData'))
def display_click(clickData):
    """Display raw click event data including custom_data."""
    if not clickData:
        return "No click event yet."
    return json.dumps(clickData, indent=2, ensure_ascii=False)


@app.callback(Output('selected-data', 'children'), Input('scatter-plot', 'selectedData'))
def display_selected(selectedData):
    """Display raw selected (box/lasso) event data."""
    if not selectedData:
        return "No points selected yet."
    return json.dumps(selectedData, indent=2, ensure_ascii=False)


@app.callback(
    Output('full-info', 'children'),
    Input('scatter-plot', 'hoverData'),
    Input('scatter-plot', 'clickData'),
    Input('scatter-plot', 'selectedData')
)
def display_readable_info(hoverData, clickData, selectedData):
    """
    Extract and display clean information from events.
    This is where we safely read custom_data.
    """
    items = []
    
    def extract_point_info(point):
        """Safely extract information from a point in event data."""
        custom = point.get('customdata', [])
        return {
            'display_name': point.get('hovertext') or point.get('text', 'Unknown'),
            'id': custom[0] if custom and len(custom) > 0 else None,
            'name': custom[1] if len(custom) > 1 else None
        }
    
    # Hover
    if hoverData and hoverData.get('points'):
        info = extract_point_info(hoverData['points'][0])
        items.append(html.P([
            html.Strong("Hover → "), 
            f"{info['display_name']} | ID: {info['id']}"
        ]))
    
    # Click
    if clickData and clickData.get('points'):
        info = extract_point_info(clickData['points'][0])
        items.append(html.P([
            html.Strong("Click → "), 
            f"{info['display_name']} | ID: {info['id']}"
        ]))
    
    # Selected (multiple points)
    if selectedData and selectedData.get('points'):
        points_list = [extract_point_info(p)['display_name'] for p in selectedData['points']]
        items.append(html.P([
            html.Strong("Selected → "), 
            ", ".join(points_list)
        ]))
    
    return items if items else html.P("No interaction yet.")


if __name__ == '__main__':
    app.run(debug=True,port=9873)